AHL-D baseline scored on the AHL-D test split. Needed to measure forgetting after fine-tuning.

In [ ]:
#@title AHL-D 29K Setup, Dataset, Vocabulary, Loaders, and Shared Helpers

!pip install -q transformers datasets huggingface_hub evaluate jiwer pillow opencv-python sentencepiece accelerate

from google.colab import drive

import os
import random
import json
import time
import copy
import math

import numpy as np
import pandas as pd

from PIL import Image, ImageOps, ImageEnhance, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from jiwer import cer, wer


drive.mount("/content/drive")


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


RESEARCH_ROOT = "/content/drive/MyDrive/amharic_htr_research"

OCR_ROOT = f"{RESEARCH_ROOT}/ocr"
OCR_RESEARCH_ROOT = f"{OCR_ROOT}/research"
SHARED_ROOT = f"{OCR_ROOT}/shared"
DATA_ROOT = f"{SHARED_ROOT}/data"
STATE_ROOT = f"{SHARED_ROOT}/state"

LOCAL_ROOT = "/content/amharic_htr"
LOCAL_DATA_ROOT = f"{LOCAL_ROOT}/data"


RESEARCH_DIRS = [
    f"{OCR_RESEARCH_ROOT}/crnn/extended",
    f"{OCR_RESEARCH_ROOT}/crnn/augmentation",
    f"{OCR_RESEARCH_ROOT}/crnn/beam_search",
    f"{OCR_RESEARCH_ROOT}/transformer/vit_htr",
    f"{OCR_RESEARCH_ROOT}/transformer/vit_augmentation",
    f"{OCR_RESEARCH_ROOT}/trocr/finetuning",
    f"{OCR_RESEARCH_ROOT}/hybrid/cnn_transformer",
    f"{OCR_RESEARCH_ROOT}/encoder_decoder/attention",
    f"{OCR_RESEARCH_ROOT}/cross_script",
    f"{OCR_RESEARCH_ROOT}/final_evaluation"
]


for directory in RESEARCH_DIRS:
    os.makedirs(
        directory,
        exist_ok=True
    )

os.makedirs(
    DATA_ROOT,
    exist_ok=True
)

os.makedirs(
    STATE_ROOT,
    exist_ok=True
)

os.makedirs(
    LOCAL_DATA_ROOT,
    exist_ok=True
)


dataset = load_dataset(
    "misiker/AHLD-29k"
)


train_raw = dataset["train"]
val_raw = dataset["validation"]
test_raw = dataset["test"]


print(
    f"Dataset loaded | "
    f"{len(train_raw)} train | "
    f"{len(val_raw)} val | "
    f"{len(test_raw)} test"
)


train_writers = set(
    train_raw["writer_id"]
)

val_writers = set(
    val_raw["writer_id"]
)

test_writers = set(
    test_raw["writer_id"]
)


print(
    f"Writers | "
    f"{len(train_writers)} train | "
    f"{len(val_writers)} val | "
    f"{len(test_writers)} test"
)


vocab_path = f"{DATA_ROOT}/ahld29k_vocabulary.json"


if os.path.exists(vocab_path):

    with open(
        vocab_path,
        "r",
        encoding="utf-8"
    ) as f:

        vocabulary = json.load(f)

    characters = vocabulary["characters"]

    blank_idx = vocabulary["blank_idx"]

    char_to_idx = {
        char: int(index)
        for char, index in vocabulary[
            "char_to_idx"
        ].items()
    }

    idx_to_char = {
        int(index): char
        for index, char in vocabulary[
            "idx_to_char"
        ].items()
    }


else:

    characters = sorted(
        set(
            "".join(
                train_raw["text"]
            )
        )
    )

    blank_idx = 0

    char_to_idx = {
        char: i + 1
        for i, char in enumerate(characters)
    }

    idx_to_char = {
        i: char
        for char, i in char_to_idx.items()
    }

    vocabulary = {
        "characters": characters,
        "blank_idx": blank_idx,
        "char_to_idx": char_to_idx,
        "idx_to_char": {
            str(i): char
            for i, char in idx_to_char.items()
        }
    }

    with open(
        vocab_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            vocabulary,
            f,
            ensure_ascii=False,
            indent=2
        )


IMG_HEIGHT = 64
IMG_WIDTH = 512
BATCH_SIZE = 16


class AHLDataset(Dataset):

    def __init__(
        self,
        dataset_split,
        augment=False
    ):

        self.dataset = dataset_split
        self.augment = augment

    def __len__(self):

        return len(self.dataset)

    def __getitem__(self, idx):

        row = self.dataset[idx]

        image = row["image"].convert("L")

        if self.augment:

            if random.random() < 0.35:

                image = ImageEnhance.Contrast(
                    image
                ).enhance(
                    random.uniform(
                        0.7,
                        1.3
                    )
                )

            if random.random() < 0.25:

                image = ImageEnhance.Brightness(
                    image
                ).enhance(
                    random.uniform(
                        0.8,
                        1.2
                    )
                )

            if random.random() < 0.20:

                image = ImageOps.autocontrast(
                    image
                )

        image = image.resize(
            (
                IMG_WIDTH,
                IMG_HEIGHT
            )
        )

        image = np.array(
            image,
            dtype=np.float32
        ) / 255.0

        image = torch.tensor(
            image
        ).unsqueeze(0)

        target = torch.tensor(
            [
                char_to_idx[c]
                for c in row["text"]
                if c in char_to_idx
            ],
            dtype=torch.long
        )

        return image, target


def collate_fn(batch):

    images = torch.stack(
        [x[0] for x in batch]
    )

    targets = torch.cat(
        [x[1] for x in batch]
    )

    target_lengths = torch.tensor(
        [
            len(x[1])
            for x in batch
        ],
        dtype=torch.long
    )

    return (
        images,
        targets,
        target_lengths
    )


train_dataset = AHLDataset(
    train_raw,
    augment=False
)

val_dataset = AHLDataset(
    val_raw,
    augment=False
)

test_dataset = AHLDataset(
    test_raw,
    augment=False
)

aug_train_dataset = AHLDataset(
    train_raw,
    augment=True
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

aug_train_loader = DataLoader(
    aug_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


def greedy_decode(outputs):

    predictions = outputs.argmax(
        dim=2
    )

    results = []

    for sequence in predictions.permute(
        1,
        0
    ):

        text = []

        previous = blank_idx

        for index in sequence.tolist():

            if (
                index != blank_idx
                and index != previous
            ):

                text.append(
                    idx_to_char.get(
                        index,
                        ""
                    )
                )

            previous = index

        results.append(
            "".join(text)
        )

    return results


def decode_targets(
    targets,
    target_lengths
):

    results = []

    offset = 0

    for length in target_lengths.tolist():

        target = targets[
            offset:offset + length
        ]

        results.append(
            "".join(
                idx_to_char.get(
                    value.item(),
                    ""
                )
                for value in target
            )
        )

        offset += length

    return results


def evaluate_model(
    model,
    loader,
    decoder=greedy_decode
):

    model.eval()

    predictions = []
    references = []

    start_time = time.time()

    with torch.no_grad():

        for (
            images,
            targets,
            target_lengths
        ) in loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            outputs = model(
                images
            )

            predictions.extend(
                decoder(outputs)
            )

            references.extend(
                decode_targets(
                    targets,
                    target_lengths
                )
            )

    elapsed = (
        time.time()
        - start_time
    )

    evaluation_cer = cer(
        references,
        predictions
    )

    evaluation_wer = wer(
        references,
        predictions
    )

    exact_accuracy = (
        sum(
            r == p
            for r, p in zip(
                references,
                predictions
            )
        )
        / len(references)
        if references
        else 0.0
    )

    return {
        "cer": float(
            evaluation_cer
        ),
        "wer": float(
            evaluation_wer
        ),
        "exact_accuracy": float(
            exact_accuracy
        ),
        "references": references,
        "predictions": predictions,
        "evaluation_time_seconds": float(
            elapsed
        ),
        "samples_per_second": float(
            len(references) / elapsed
            if elapsed > 0
            else 0
        )
    }


def save_predictions(
    references,
    predictions,
    output_path
):

    pd.DataFrame({
        "sample_index": range(
            len(references)
        ),
        "ground_truth": references,
        "ocr_prediction": predictions
    }).to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )


def save_metrics(
    metrics,
    output_path
):

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            metrics,
            f,
            ensure_ascii=False,
            indent=2
        )


with open(
    f"{DATA_ROOT}/ahld29k_dataset_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "dataset": "misiker/AHLD-29k",
            "image_height": IMG_HEIGHT,
            "image_width": IMG_WIDTH,
            "batch_size": BATCH_SIZE,
            "vocabulary_size": len(characters),
            "training_samples": len(train_raw),
            "validation_samples": len(val_raw),
            "test_samples": len(test_raw),
            "training_writers": len(train_writers),
            "validation_writers": len(val_writers),
            "test_writers": len(test_writers),
            "vocabulary": vocab_path
        },
        f,
        ensure_ascii=False,
        indent=2
    )


with open(
    f"{STATE_ROOT}/ahld29k_runtime_state.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "seed": SEED,
            "device": str(DEVICE),
            "gpu": (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else None
            ),
            "torch_version": torch.__version__,
            "research_root": RESEARCH_ROOT
        },
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    f"Ready | {DEVICE} | "
    f"{len(train_raw)} train | "
    f"{len(val_raw)} val | "
    f"{len(test_raw)} test | "
    f"{len(characters)} chars"
)

In [ ]:
#@title AHL-D CRNN baseline on the AHL-D test split

class CRNN(nn.Module):

    def __init__(
        self,
        num_classes
    ):

        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(
                1,
                64,
                3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(
                64,
                128,
                3,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(
                128,
                256,
                3,
                padding=1
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                256,
                256,
                3,
                padding=1
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(
                kernel_size=(2, 1),
                stride=(2, 1)
            ),

            nn.Conv2d(
                256,
                512,
                3,
                padding=1
            ),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                512,
                512,
                3,
                padding=1
            ),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(
                kernel_size=(2, 1),
                stride=(2, 1)
            )
        )

        self.rnn = nn.LSTM(
            input_size=512,
            hidden_size=256,
            num_layers=2,
            bidirectional=True,
            dropout=0.2
        )

        self.classifier = nn.Linear(
            512,
            num_classes
        )

    def forward(self, x):

        x = self.cnn(x)

        x = F.adaptive_avg_pool2d(
            x,
            (
                1,
                x.size(3)
            )
        )

        x = x.squeeze(2)

        x = x.permute(
            2,
            0,
            1
        )

        x, _ = self.rnn(x)

        x = self.classifier(x)

        return x


checkpoint_path = f"{OCR_RESEARCH_ROOT}/crnn/ahld29k/best_checkpoint.pt"
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

model = CRNN(len(characters) + 1).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])

result = evaluate_model(model, test_loader)

print(
    f"AHL-D test | {len(test_raw)} samples | "
    f"CER {result['cer']:.4f} | WER {result['wer']:.4f}"
)

out_dir = f"{OCR_RESEARCH_ROOT}/crnn/ahld29k"

save_predictions(
    result["references"],
    result["predictions"],
    f"{out_dir}/test_predictions.csv"
)

save_metrics(
    {
        "samples": len(test_raw),
        "cer": result["cer"],
        "wer": result["wer"],
        "exact_accuracy": result["exact_accuracy"]
    },
    f"{out_dir}/test_metrics.json"
)
